[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PARODBE/UAX_Fundamento_Ciencia_de_Datos/blob/main/Notebooks/Fase_2_Organizacion_Datos.ipynb)

# Fase 2. Organización y entendimiento de los datos

En esta fase preparo el conjunto de datos que usaré en las siguientes partes. Leo DILIst y una tabla compacta de asociaciones de nombres con estructuras. Calculo los descriptores moleculares de RDKit, reviso la calidad de las coincidencias y guardo los CSV de entrenamiento y prueba. La unidad de análisis será una estructura química identificada por su InChIKey.

Uso DILIst como fuente de etiquetas de lesión hepática. Para cada nombre busco la estructura en PubChem mediante PubChemPy. Solo acepto una coincidencia cuando el resultado tiene un único InChIKey. Conservo el origen y el estado de cada búsqueda en el archivo de estructuras.

In [ ]:
# Instalo las librerías que necesito para las consultas y el cálculo molecular.
%pip -q install rdkit pubchempy

# Importo las herramientas para leer, consultar y preparar los datos.
from pathlib import Path  # Organizo las rutas de salida.
import subprocess  # Clono el proyecto si aún no está en Colab.
import pandas as pd  # Trabajo con tablas.
import numpy as np  # Uso valores numéricos y nulos.
import pubchempy as pcp  # Busco estructuras por nombre en PubChem.
from rdkit import Chem  # Convierto SMILES en moléculas.
from rdkit.Chem import Descriptors  # Accedo al catálogo de descriptores RDKit.
from IPython.display import display  # Muestro tablas en Colab.

# Clono el repositorio para guardar los archivos generados dentro de Data.
REPO_DIR = Path('/content/UAX_Fundamento_Ciencia_de_Datos')  # Defino la carpeta local del proyecto.
if not REPO_DIR.exists():  # Compruebo si ya está clonado en esta sesión.
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/PARODBE/UAX_Fundamento_Ciencia_de_Datos.git', str(REPO_DIR)], check=True)  # Descargo el repositorio.
DATA_DIR = REPO_DIR / 'Data'  # Guardo los archivos generados en Data.
DATA_DIR.mkdir(parents=True, exist_ok=True)  # Creo la carpeta si hace falta.

# Defino dónde están los archivos de partida.
URL_DILIST = DATA_DIR / 'DILIst_FDA.csv'  # Leo las etiquetas originales.
import time  # Espero un poco entre consultas a PubChem.


In [ ]:
# Cargo DILIst desde el CSV público del proyecto.
dilist_original = pd.read_csv(URL_DILIST)  # Leo las etiquetas originales.
dilist_original.to_csv(DATA_DIR / 'DILIst_FDA.csv', index=False)  # Guardo una copia dentro de Data.

# Compruebo el tamaño y los tipos de datos de la fuente.
print('Filas y columnas de DILIst:', dilist_original.shape)  # Informo las dimensiones.
display(dilist_original.dtypes.astype(str).rename('tipo de dato').to_frame())  # Muestro el tipo de cada columna.
display(dilist_original.head())  # Reviso los primeros registros.

# Describo los campos que utilizaré más adelante.
descripcion = pd.DataFrame({'variable': ['DILIST_ID', 'CompoundName', 'DILIst_Classification', 'Routes_of_Administration'], 'descripción': ['Identificador del registro FDA', 'Nombre del compuesto', 'Etiqueta DILI: 0 negativa y 1 positiva', 'Vía de administración registrada en la fuente']})  # Creo un diccionario breve.
display(descripcion)  # Muestro la descripción de las columnas.

# Aclaro qué representa una fila y la limitación temporal de la fuente.
print('Unidad de análisis: un registro de fármaco de DILIst; la estructura se identificará por InChIKey cuando esté disponible.')  # Describo la unidad.
print('Rango temporal: no disponible; la fuente no contiene una fecha por registro.')  # Informo la limitación temporal.


## 2.3. Obtención de estructuras químicas

Busco los nombres de DILIst en PubChem con PubChemPy. Cuando hay resultados, tomo el primero, guardo su SMILES y su InChIKey, y elimino posibles duplicados por InChIKey. El CSV de salida contiene únicamente los compuestos encontrados.

In [ ]:
# Busco las estructuras de los compuestos en PubChem.
archivo_estructuras = DATA_DIR / 'estructuras_nombres.csv'  # Defino el archivo de salida.
nombres = dilist_original['CompoundName'].dropna().astype(str).str.strip().drop_duplicates()  # Tomo los nombres únicos.
estructuras = []  # Guardo solo los compuestos encontrados.

for numero, nombre in enumerate(nombres, start=1):  # Recorro los nombres de DILIst.
    candidatos = []  # Inicio la búsqueda sin resultados.
    for intento in range(3):  # Reintento si PubChem tiene un fallo temporal.
        try:  # Consulto PubChem mediante PubChemPy.
            candidatos = pcp.get_compounds(nombre, 'name')  # Busco el compuesto por nombre.
            break  # Salgo del bucle si la consulta responde.
        except Exception:  # Si falla la consulta, espero antes de reintentar.
            time.sleep(1 + intento)  # Aumento ligeramente la pausa en cada reintento.
    if candidatos:  # Continúo si PubChem devolvió algún compuesto.
        compuesto = candidatos[0]  # Tomo el primer resultado de PubChem.
        smiles = compuesto.canonical_smiles or compuesto.isomeric_smiles  # Obtengo el SMILES disponible.
        if compuesto.inchikey and smiles:  # Guardo solo registros con identificador y estructura.
            estructuras.append({'CompoundName': nombre, 'InChIKey': compuesto.inchikey, 'SMILES': smiles})  # Añado el compuesto.
    if numero % 50 == 0:  # Guardo avances durante la consulta.
        pd.DataFrame(estructuras).drop_duplicates('InChIKey').to_csv(archivo_estructuras, index=False)  # Actualizo el CSV.
    time.sleep(0.2)  # Dejo una pausa breve entre consultas.

# Quito duplicados y guardo solo los compuestos encontrados.
estructuras = pd.DataFrame(estructuras).drop_duplicates('InChIKey').reset_index(drop=True)  # Dejo un registro por estructura.
estructuras.to_csv(archivo_estructuras, index=False)  # Guardo los resultados en Data.
print('Compuestos con estructura única:', len(estructuras))  # Informo cuántos compuestos se obtuvieron.
print('Archivo guardado en:', archivo_estructuras)  # Indico dónde quedó el CSV.


Normalizo los nombres solo para hacer la comparación, pero conservo el nombre original de DILIst en la tabla final. El mapa de estructuras ya incluye una fila por nombre y señala si la estructura quedó identificada, ambigua o sin coincidencia.

In [ ]:
# Renombro la etiqueta para trabajar con un nombre corto y uniforme.
dilist = dilist_original.rename(columns={'DILIst_Classification': 'DILI'}).copy()  # Copio los datos y renombro la clase.
dilist['CompoundName'] = dilist['CompoundName'].astype(str).str.strip()  # Quito espacios sobrantes del nombre.
dilist['DILI'] = pd.to_numeric(dilist['DILI'], errors='coerce')  # Convierto la etiqueta a número.
dilist = dilist.dropna(subset=['CompoundName', 'DILI']).copy()  # Retiro filas sin nombre o clase.

# Preparo claves normalizadas en las dos tablas para unir nombres sin distinguir mayúsculas.
dilist['nombre_clave'] = dilist['CompoundName'].str.upper()  # Normalizo nombres de DILIst.
estructuras['nombre_clave'] = estructuras['CompoundName'].astype(str).str.strip().str.upper()  # Normalizo nombres del mapa.
estructuras = estructuras.drop(columns='CompoundName')  # Quito el nombre duplicado del mapa.
datos = dilist.merge(estructuras, on='nombre_clave', how='left')  # Uno las etiquetas y las estructuras.
datos = datos.drop(columns='nombre_clave')  # Quito la clave auxiliar de trabajo.

# Reviso cuántas filas tienen una estructura inequívoca.
print('Registros DILIst:', len(datos))  # Muestro el total de registros.
print('Con InChIKey:', datos['InChIKey'].notna().sum())  # Cuento las estructuras resueltas.
print('Sin estructura inequívoca:', datos['InChIKey'].isna().sum())  # Cuento los casos pendientes.


Calculo todos los descriptores escalares que trae el catálogo de RDKit (en esta versión, alrededor de 200). Así no reduzco de antemano la información química disponible. Para el EDA describo cinco propiedades fáciles de interpretar —las cuatro de Lipinski y TPSA—; en los modelos selecciono entre el conjunto amplio y retiro variables constantes o muy correlacionadas usando solo entrenamiento. No calculo fingerprints.

In [ ]:
# Defino una función que calcula todos los descriptores escalares disponibles en RDKit.
def calcular_descriptores(smiles):  # Recibo una cadena SMILES.
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) and smiles else None  # Creo la molécula si hay SMILES.
    if mol is None:  # Compruebo que RDKit haya leído la estructura.
        return {}  # Devuelvo vacío si no puedo procesarla.
    valores = {}  # Creo un diccionario para las propiedades moleculares.
    for nombre, funcion in Descriptors._descList:  # Recorro el catálogo de descriptores RDKit.
        try:  # Intento calcular cada propiedad.
            valores[nombre] = funcion(mol)  # Guardo el resultado calculado.
        except Exception:  # Evito que un descriptor aislado detenga todo el cálculo.
            valores[nombre] = np.nan  # Marco como ausente el valor que no pude calcular.
    return valores  # Devuelvo todos los descriptores calculados.

# Calculo una fila de propiedades para cada SMILES.
lista_descriptores = [calcular_descriptores(smiles) for smiles in datos['SMILES']]  # Aplico la función a las estructuras.
descriptores = pd.DataFrame(lista_descriptores)  # Convierto los resultados en una tabla.
datos = pd.concat([datos.reset_index(drop=True), descriptores], axis=1)  # Uno los descriptores con las etiquetas.

# Cuento incumplimientos de los cuatro límites de Lipinski.
datos['LipinskiViolations'] = ((datos['MolWt'] >= 500).astype(int) + (datos['MolLogP'] > 5).astype(int) + (datos['NumHDonors'] > 5).astype(int) + (datos['NumHAcceptors'] > 10).astype(int))  # Sumo los cuatro criterios.

# Muestro algunas variables químicas para revisar el cálculo.
columnas_demo = ['CompoundName', 'DILI', 'InChIKey', 'MolWt', 'MolLogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']  # Selecciono columnas legibles.
display(datos[columnas_demo].head())  # Muestro una muestra de los resultados.
print('Descriptores RDKit calculados:', len(descriptores.columns))  # Informo cuántas propiedades se calcularon.


## 2.4. Calidad, duplicados y partición

Retiro de los análisis moleculares los registros sin estructura inequívoca. Si la misma InChIKey aparece con etiquetas DILI diferentes, la separo para revisión. Después dejo una fila por InChIKey y creo una partición 80/20. Estratifico por cuantiles de MolLogP y, cuando el tamaño de los grupos lo permite, también por la clase DILI.

In [ ]:
# Normalizo la InChIKey y separo los registros con estructura identificada.
datos['InChIKey'] = datos['InChIKey'].astype('string').str.upper()  # Normalizo el identificador químico.
con_etiqueta = datos.dropna(subset=['InChIKey']).copy()  # Conservo moléculas con estructura.

# Identifico claves químicas que aparecen con etiquetas distintas.
clases_por_clave = con_etiqueta.groupby('InChIKey')['DILI'].nunique()  # Cuento etiquetas distintas por estructura.
claves_conflictivas = clases_por_clave[clases_por_clave > 1].index  # Selecciono estructuras con conflicto.
datos_conflicto = con_etiqueta[con_etiqueta['InChIKey'].isin(claves_conflictivas)].copy()  # Guardo esos casos aparte.
datos_limpios = con_etiqueta[~con_etiqueta['InChIKey'].isin(claves_conflictivas)].drop_duplicates('InChIKey').reset_index(drop=True)  # Dejo una fila por estructura válida.

# Busco un número de cuantiles que permita estratificar sin grupos demasiado pequeños.
estratos = None  # Inicio sin una estratificación elegida.
for numero_cuantiles in range(10, 1, -1):  # Pruebo de diez cuantiles a dos.
    cuantiles = pd.qcut(datos_limpios['MolLogP'], q=numero_cuantiles, duplicates='drop')  # Agrupo MolLogP por cuantiles.
    propuesta = datos_limpios['DILI'].astype(int).astype(str) + '_' + cuantiles.astype(str)  # Combino cuantiles y clase.
    if propuesta.value_counts().min() >= 2:  # Compruebo que cada grupo permita dividirse.
        estratos = propuesta  # Guardo la estratificación más detallada disponible.
        break  # Termino la búsqueda.
if estratos is None:  # Compruebo si no fue posible combinar con la clase.
    estratos = cuantiles if cuantiles.value_counts().min() >= 2 else None  # Uso solo cuantiles si es posible.

# Creo la partición fija para que los resultados sean reproducibles.
from sklearn.model_selection import train_test_split  # Importo la función de partición.
train_df, test_df = train_test_split(datos_limpios, test_size=0.20, random_state=42, stratify=estratos)  # Separo 80 % y 20 %.
print('Train:', train_df.shape, '| Test:', test_df.shape)  # Muestro el tamaño de cada parte.
print('Clases en train:', train_df['DILI'].value_counts(normalize=True).round(3).to_dict())  # Reviso la proporción de clases en train.
print('Clases en test:', test_df['DILI'].value_counts(normalize=True).round(3).to_dict())  # Reviso la proporción de clases en test.


## 2.5. Exportación de los CSV

Guardo en la carpeta Data de la copia del proyecto los CSV de partida y los archivos que genero para las fases siguientes.

In [ ]:
# Defino el conjunto de descriptores numéricos calculados, salvo columnas de identificación y resultado.
columnas_excluir = ['DILI', 'DILIST_ID', 'InChIKey', 'SMILES', 'CompoundName', 'Routes_of_Administration', 'chembl_id']  # Evito incluir etiquetas e identificadores como propiedades.
DESCRIPTORES = [columna for columna in descriptores.select_dtypes(include='number').columns if columna not in columnas_excluir]  # Selecciono propiedades numéricas.
DESCRIPTORES = [columna for columna in DESCRIPTORES if columna in datos_limpios.columns]  # Conservo solo columnas presentes en los datos depurados.

# Guardo únicamente las tablas que utilizaré en las fases siguientes.
train_df.to_csv(DATA_DIR / 'dilist_train.csv', index=False)  # Guardo la partición de entrenamiento.
test_df.to_csv(DATA_DIR / 'dilist_test.csv', index=False)  # Guardo la partición de prueba.
datos_limpios.to_csv(DATA_DIR / 'dilist_descriptores.csv', index=False)  # Guardo los datos moleculares depurados.

# Informo dónde quedaron los archivos preparados.
print('He guardado los CSV en:', DATA_DIR)  # Muestro la carpeta de salida.
print('Descriptores numéricos disponibles para los modelos:', len(DESCRIPTORES))  # Muestro el número de propiedades disponibles.


## Interpretación de los datos preparados

El recuento anterior indica cuántos compuestos de DILIst han quedado asociados a una estructura de PubChem, después de eliminar duplicados por InChIKey. El resto del notebook utiliza esa tabla para calcular descriptores y preparar los conjuntos de entrenamiento y prueba.

## Descargar los archivos generados

Al terminar, preparo un ZIP con los CSV generados para descargarlos desde Colab y añadirlos manualmente a `Data` en GitHub.

In [ ]:
# Preparo juntos los CSV que genera esta fase.
from pathlib import Path  # Trabajo con las rutas de los archivos.
from zipfile import ZipFile  # Creo un archivo comprimido.
from google.colab import files  # Descargo el ZIP desde Colab.

archivos_generados = ['estructuras_nombres.csv', 'dilist_descriptores.csv', 'dilist_train.csv', 'dilist_test.csv']  # Selecciono los resultados.
zip_salida = Path('/content/resultados_fase2.zip')  # Guardo el ZIP fuera de Data.
with ZipFile(zip_salida, 'w') as zip_fase2:  # Creo el archivo comprimido.
    for nombre in archivos_generados:  # Recorro cada resultado.
        zip_fase2.write(DATA_DIR / nombre, arcname=nombre)  # Lo añado al ZIP.
files.download(str(zip_salida))  # Inicio la descarga a mi equipo.
